In [1]:
"""
Print system specifications as an Attribute / Value table.

Works on Linux, Windows and macOS. Optional packages improve results:
    pip install psutil distro
GPU / CUDA / driver info needs an NVIDIA GPU with nvidia-smi on PATH.
"""

import os
import platform
import re
import subprocess
import sys


def run(cmd):
    """Run a command and return its stdout, or '' if it fails."""
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=10,
                             shell=isinstance(cmd, str))
        return out.stdout.strip()
    except Exception:
        return ""


def gb(n_bytes):
    return f"{round(n_bytes / 1024 ** 3)} GB"


# ---------------------------------------------------------------- OS
def get_os():
    system = platform.system()
    if system == "Linux":
        try:
            import distro
            name = distro.name(pretty=True)
            if name:
                return name
        except ImportError:
            pass
        try:
            with open("/etc/os-release") as f:
                for line in f:
                    if line.startswith("PRETTY_NAME="):
                        return line.split("=", 1)[1].strip().strip('"')
        except OSError:
            pass
    if system == "Darwin":
        return f"macOS {platform.mac_ver()[0]}"
    return f"{system} {platform.release()} ({platform.version()})"


# ---------------------------------------------------------------- CPU
def get_cpu_model():
    system = platform.system()
    if system == "Linux":
        try:
            with open("/proc/cpuinfo") as f:
                for line in f:
                    if line.startswith("model name"):
                        return line.split(":", 1)[1].strip()
        except OSError:
            pass
    elif system == "Windows":
        out = run('powershell -NoProfile -Command "(Get-CimInstance Win32_Processor).Name"')
        if out:
            return out.splitlines()[0].strip()
    elif system == "Darwin":
        out = run(["sysctl", "-n", "machdep.cpu.brand_string"])
        if out:
            return out
    return platform.processor() or "N/A"


def get_cpu_cores():
    return str(os.cpu_count() or "N/A")  # logical cores, as in the example


# ---------------------------------------------------------------- Memory
def get_memory():
    try:
        import psutil
        return gb(psutil.virtual_memory().total), gb(psutil.swap_memory().total)
    except ImportError:
        pass
    if platform.system() == "Linux":
        info = {}
        with open("/proc/meminfo") as f:
            for line in f:
                key, val = line.split(":", 1)
                info[key] = int(val.split()[0]) * 1024  # kB -> bytes
        return gb(info.get("MemTotal", 0)), gb(info.get("SwapTotal", 0))
    return "N/A (pip install psutil)", "N/A (pip install psutil)"


# ---------------------------------------------------------------- GPU / NVIDIA
def get_gpu_and_driver():
    out = run(["nvidia-smi", "--query-gpu=name,driver_version",
               "--format=csv,noheader"])
    if not out:
        return "N/A", "N/A"
    names, drivers = [], []
    for line in out.splitlines():
        name, driver = [x.strip() for x in line.split(",")]
        names.append(name)
        drivers.append(driver)
    return ", ".join(names), drivers[0]


def get_cuda_version():
    # Toolkit version (nvcc) is what you compiled against
    out = run(["nvcc", "--version"])
    m = re.search(r"release (\d+\.\d+)", out)
    if m:
        return m.group(1)
    # Fall back to the max CUDA version the driver supports
    out = run(["nvidia-smi"])
    m = re.search(r"CUDA Version:\s*([\d.]+)", out)
    if m:
        return m.group(1) + " (driver)"
    # Fall back to what TensorFlow was built with
    info = tf_build_info()
    return info.get("cuda_version", "N/A")


def get_cudnn_version():
    # Look for cudnn header files
    for path in ["/usr/include/cudnn_version.h", "/usr/include/cudnn.h",
                 "/usr/local/cuda/include/cudnn_version.h",
                 "/usr/local/cuda/include/cudnn.h"]:
        try:
            with open(path) as f:
                text = f.read()
            parts = [re.search(rf"#define CUDNN_{k}\s+(\d+)", text)
                     for k in ("MAJOR", "MINOR", "PATCHLEVEL")]
            if all(parts):
                return ".".join(p.group(1) for p in parts)
        except OSError:
            continue
    # Fall back to TensorFlow / PyTorch build info
    info = tf_build_info()
    if info.get("cudnn_version"):
        return str(info["cudnn_version"])
    try:
        import torch
        v = torch.backends.cudnn.version()
        if v:
            return f"{v // 1000}.{(v % 1000) // 100}.{v % 100}"
    except Exception:
        pass
    return "N/A"


_tf_info = None


def tf_build_info():
    global _tf_info
    if _tf_info is None:
        _tf_info = {}
        try:
            import tensorflow as tf
            _tf_info = dict(tf.sysconfig.get_build_info())
        except Exception:
            pass
    return _tf_info


# ---------------------------------------------------------------- Compiler / libs
def get_gcc_version():
    out = run(["gcc", "--version"])
    if out:
        first = out.splitlines()[0]
        return first.split(" ", 1)[1] if first.startswith("gcc") else first
    return "N/A"


def get_package_version(module_name):
    try:
        module = __import__(module_name)
        return module.__version__
    except Exception:
        return "Not installed"


# ---------------------------------------------------------------- Main
def collect():
    memory, swap = get_memory()
    gpu, driver = get_gpu_and_driver()
    return [
        ("OS", get_os()),
        ("CPU model name", get_cpu_model()),
        ("CPU cores", get_cpu_cores()),
        ("Memory", memory),
        ("Swap", swap),
        ("GPU model name", gpu),
        ("CUDA version", get_cuda_version()),
        ("cuDNN version", get_cudnn_version()),
        ("Nvidia driver version", driver),
        ("GCC version", get_gcc_version()),
        ("Python version", platform.python_version()),
        ("Tensorflow version", get_package_version("tensorflow")),
        ("Numpy version", get_package_version("numpy")),
    ]


def print_table(rows):
    w1 = max(len(r[0]) for r in rows + [("Attribute", "")])
    w2 = max(len(str(r[1])) for r in rows + [("", "Value")])
    line = "-" * (w1 + w2 + 3)
    print(line)
    print(f"{'Attribute':<{w1}} | {'Value':^{w2}}")
    print(line)
    for k, v in rows:
        print(f"{k:<{w1}} | {str(v):^{w2}}")
    print(line)


In [2]:
rows = collect()
print_table(rows)

----------------------------------------------------------
Attribute             |               Value               
----------------------------------------------------------
OS                    |      Windows 11 (10.0.26300)      
CPU model name        |   Intel(R) Core(TM) Ultra 9 285H  
CPU cores             |                 16                
Memory                |               31 GB               
Swap                  |               38 GB               
GPU model name        | NVIDIA GeForce RTX 5070 Laptop GPU
CUDA version          |                N/A                
cuDNN version         |               92.4.0              
Nvidia driver version |               616.56              
GCC version           |                N/A                
Python version        |               3.12.0              
Tensorflow version    |           Not installed           
Numpy version         |               1.26.4              
--------------------------------------------------------